# Trend-Filtered Basis Arbitrage: PAXG (Spot Gold) vs. Gold Perpetual Futures

This notebook combines two classic, individually well-known trading ideas and tests whether
combining them beats either alone:

1. **Trend detection** (trend lines / rolling regression slope) — is the underlying asset trending,
   and how strongly?
2. **Historical-average deviation arbitrage** (mean reversion / stat-arb) — when a spread between two
   related instruments deviates far from its own rolling historical average, bet on it reverting.

**The pair**: **PAXG** (Pax Gold — an ERC-20 token, each unit redeemable for one fine troy ounce of
physical gold held in a vault; effectively "spot gold" with on-chain settlement) vs. **XAU-USDT-SWAP**
(OKX's gold perpetual futures contract — a synthetic derivative on gold's USD price, settled and
funded like any crypto perpetual). Both track the same underlying (gold), but through **completely
different mechanisms** — physical backing + redemption arbitrage for PAXG, vs. synthetic
derivative + funding-rate mechanics for the perpetual. That structural difference is exactly what
should create a genuine, tradeable basis (price gap) between them, and exactly why *both* trend and
mean-reversion might matter: does the underlying's trend systematically bias the basis (because
leveraged perpetual positioning tends to chase momentum harder than a physically-redeemable token
does), such that a naive "always fade the extreme" arbitrage gets run over during trends?

**The actual research question this notebook answers empirically** (not just assumed): *does the
gold trend predict/explain basis deviation?* If yes, filtering the mean-reversion arb by trend
regime should improve it. If no, the two ideas don't actually combine — they're just decoration on
top of each other. We test this directly (Section 4) before building or backtesting anything, so
the strategy that follows is evidence-based, not an assumed synergy.

**Data source**: OKX's public market-data API (free, no key required) for both instruments, since
they trade on the same exchange (avoids cross-exchange timestamp/data-quality mismatches). Two real
constraints on how much confidence to place in what follows: XAU-USDT-SWAP's history only reaches
back to ~April 2025, but **PAXG-USDT's listing on OKX specifically is even newer** (~October 2025) —
the two only *overlap* for about 10 months, which is what actually bounds every analysis below (the
individual instruments' own histories are longer, but the arbitrage only exists where both exist).
Treat everything here as illustrative research on a short sample, not a validated edge.

> ⚠️ Research only, not investment advice. This notebook models a market-neutral spread position
> (long one leg, short the other) with no transaction costs, funding payments, borrow costs, or
> execution slippage — all of which matter a lot in a real basis trade at this timeframe.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection

OKX's `/market/candles` endpoint gives the most recent ~300 daily candles; `/market/history-candles`
paginates further back via an `after` cursor. We combine both to get the full available history for
each instrument, drop the most recent *unconfirmed* (still-forming) candle each returns, and dedupe.


In [ ]:
OKX_BASE = "https://www.okx.com/api/v5/market"


def fetch_okx_full_history(inst_id, bar="1D"):
    all_rows = []

    r = requests.get(f"{OKX_BASE}/candles", params={"instId": inst_id, "bar": bar, "limit": 300}, timeout=20)
    r.raise_for_status()
    data = r.json()["data"]
    all_rows.extend(data)
    after = min(int(row[0]) for row in data)

    while True:
        r = requests.get(
            f"{OKX_BASE}/history-candles",
            params={"instId": inst_id, "bar": bar, "limit": 100, "after": after},
            timeout=20,
        )
        r.raise_for_status()
        data = r.json()["data"]
        if not data:
            break
        all_rows.extend(data)
        after = min(int(row[0]) for row in data)
        time.sleep(0.25)

    cols = ["ts", "open", "high", "low", "close", "vol", "volCcy", "volCcyQuote", "confirm"]
    df = pd.DataFrame(all_rows, columns=cols)
    df = df[df["confirm"] == "1"]  # drop any still-forming candle
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="ms").dt.normalize()
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "vol"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "vol"]]


paxg = fetch_okx_full_history("PAXG-USDT")
perp = fetch_okx_full_history("XAU-USDT-SWAP")

print(f"PAXG-USDT:      {paxg.index.min().date()} to {paxg.index.max().date()}, {len(paxg)} days")
print(f"XAU-USDT-SWAP:  {perp.index.min().date()} to {perp.index.max().date()}, {len(perp)} days")


In [ ]:
# Funding rate history (paginated). This is the mechanism that's SUPPOSED to pull a crypto
# perpetual's price back toward the underlying — worth checking whether it's actually doing
# anything for this specific contract before assuming it explains the basis.
def fetch_okx_funding_history(inst_id, max_pages=10):
    all_rows = []
    before = None
    for _ in range(max_pages):
        params = {"instId": inst_id, "limit": 100}
        if before:
            params["before"] = before
        r = requests.get(f"{OKX_BASE.replace('/market','/public')}/funding-rate-history", params=params, timeout=20)
        r.raise_for_status()
        data = r.json()["data"]
        if not data:
            break
        all_rows.extend(data)
        before = min(int(row["fundingTime"]) for row in data)
        time.sleep(0.25)
    df = pd.DataFrame(all_rows)
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["fundingTime"].astype(np.int64), unit="ms").dt.normalize()
    df["fundingRate"] = df["fundingRate"].astype(float)
    return df.groupby("date")["fundingRate"].mean().sort_index()


funding = fetch_okx_funding_history("XAU-USDT-SWAP")
print(f"Funding rate data: {len(funding)} days")
print(f"Non-zero funding days: {(funding != 0).sum()} / {len(funding)}")
print(f"Mean funding rate: {funding.mean():.8f}, std: {funding.std():.8f}")


## 2. Basis Construction

We align both series to their overlapping trading history and define the **basis** as the
perpetual's premium/discount relative to PAXG, in percent — using percent rather than raw dollars
since gold's own price level moved substantially over the sample and we want a scale-free measure of
mispricing.


In [ ]:
df = pd.DataFrame({
    "paxg_close": paxg["close"],
    "perp_close": perp["close"],
}).dropna()

df["basis_pct"] = df["perp_close"] / df["paxg_close"] - 1
df["funding"] = funding.reindex(df.index)

print(f"Overlapping sample: {df.index.min().date()} to {df.index.max().date()}, {len(df)} days")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["paxg_close"], label="PAXG (spot)", color="tab:blue")
axes[0].plot(df.index, df["perp_close"], label="XAU-USDT-SWAP (perp)", color="tab:orange", alpha=0.8)
axes[0].set_title("PAXG Spot vs. Gold Perpetual Futures")
axes[0].legend()

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green")
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("Basis: (Perp / PAXG - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) — the core assumption "
      "the arbitrage leg of this strategy depends on.")


## 3. Trend Detection

Two representations of the same idea, for different purposes:

- **Geometric trend lines** (for visualization/intuition) — connect the most recent swing highs and
  swing lows with a straight line, the literal technical-analysis construct.
- **Rolling regression slope** (for the tradeable signal) — fit a line through the last *N* days of
  price and take its slope, expressed as an implied %-move-per-window. Continuous, differentiable,
  easy to backtest — a cleaner input than trying to detect discrete trendline breakouts.


In [ ]:
def find_pivots(price, window=5):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=3):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).days.values.astype(float)
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


swing_highs, swing_lows = find_pivots(df["paxg_close"], window=5)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["paxg_close"], color="tab:blue", label="PAXG close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=25, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=25, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, df.index[-1])
        y_line = intercept + slope * (x_line - x0).days
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("PAXG Price with Recent Swing-High / Swing-Low Trend Lines")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def rolling_trend_pct(price, window=20):
    def slope(y):
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    raw_slope = price.rolling(window).apply(slope, raw=True)
    return raw_slope * window / price  # implied % move over the window, scale-free

df["trend_pct"] = rolling_trend_pct(df["paxg_close"], window=20)

fig, ax = plt.subplots()
df["trend_pct"].mul(100).plot(ax=ax, color="purple")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Rolling 20-Day Trend Signal (Implied %% Move Over Window)")
ax.set_ylabel("%")
plt.tight_layout()
plt.show()


## 4. Does the Trend Predict Basis Deviation? (The Actual Research Question)

Before building anything, test the hypothesis directly: is the basis's *deviation from its own
rolling average* correlated with the underlying's trend? If leveraged perpetual positioning chases
momentum harder than PAXG's slower, redemption-arbitrage-anchored price does, the perp should trade
at a **premium during uptrends** and a **discount during downtrends** — i.e. a *positive* correlation
between `trend_pct` and basis z-score.


In [ ]:
def zscore(s, window=20):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

df["basis_z"] = zscore(df["basis_pct"], window=20)

sub = df[["trend_pct", "basis_z"]].dropna()
corr_val = sub["trend_pct"].corr(sub["basis_z"])

fig, ax = plt.subplots()
ax.scatter(sub["trend_pct"] * 100, sub["basis_z"], alpha=0.5, s=20)
ax.set_xlabel("Rolling trend signal (%% move over 20d)")
ax.set_ylabel("Basis z-score")
ax.set_title(f"Trend vs. Basis Deviation (n={len(sub)}, corr={corr_val:.3f})")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"Correlation(trend_pct, basis_z) = {corr_val:.3f}")
if abs(corr_val) < 0.15:
    print("Near zero: the two ideas don't actually combine here on this sample — a trend filter "
          "wouldn't be expected to help, and any backtest improvement below should be treated "
          "skeptically as likely noise rather than a real mechanistic effect.")
elif corr_val > 0:
    print("Positive: the perp tends to richen during uptrends and cheapen during downtrends — "
          "consistent with leveraged perpetual positioning chasing momentum harder than PAXG's "
          "slower, redemption-anchored price does.")
else:
    print("Negative: the perp tends to CHEAPEN during uptrends and richen during downtrends — the "
          "opposite of the momentum-chasing hypothesis this notebook opened with. Real, but not the "
          "mechanism assumed going in — worth investigating why (e.g. does PAXG see disproportionate "
          "buying pressure specifically during gold rallies?) before leaning on it.")


## 5. Mean-Reversion Arbitrage Signal

Classic z-score entry/exit on the basis, with explicit holding-state logic (not just flipping
position on every threshold crossing) — enter when `|basis_z|` clears the entry threshold, hold
until it reverts back inside the exit threshold.

**Position convention**: `+1` = long the basis (long perp / short PAXG, taken when the perp is
*cheap* relative to its own history, betting the basis rises back toward its average). `-1` = short
the basis (short perp / long PAXG, taken when the perp is *rich*, betting the basis falls). P&L of
a dollar-neutral spread position is closely approximated by `position x change in basis_pct`.


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    """blocked: optional boolean Series — True blocks NEW entries (but not exits) on that day."""
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
df["basis_z"].plot(ax=axes[0], color="tab:green")
axes[0].axhline(ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(-ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].set_title("Basis Z-Score with Entry Thresholds")

position_pure_mr.plot(ax=axes[1], color="black", drawstyle="steps-post")
axes[1].set_title("Pure Mean-Reversion Position (+1 = long basis, -1 = short basis)")
plt.tight_layout()
plt.show()


## 6. Combining Both Ideas: Trend-Filtered Arbitrage

Using Section 4's finding directly: define a "strong trend" regime as `|trend_pct|` above its own
rolling 67th percentile (adaptive, not a fixed constant — avoids picking a threshold that only looks
good in hindsight on this specific sample), and **block new mean-reversion entries** while the
underlying is in a strong trend. Existing positions still exit normally.


In [ ]:
trend_strength = df["trend_pct"].abs()
trend_threshold = trend_strength.rolling(90, min_periods=30).quantile(0.67)
strong_trend = trend_strength > trend_threshold

position_trend_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=strong_trend)

print(f"Days blocked by trend filter: {strong_trend.sum()} / {strong_trend.notna().sum()} "
      f"({strong_trend.mean()*100:.1f}%)")


In [ ]:
def sharpe(returns, periods_per_year=365):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "trend_filtered_mean_reversion": position_trend_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Trend-Filtered Basis Arbitrage")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(365),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "trend_filtered_mean_reversion": (position_trend_filtered.diff().abs() > 0).sum(),
    }),
})
summary


## 7. Market-Neutrality Check

A genuine arbitrage strategy's returns should be roughly **uncorrelated with gold's own direction**
— that's the entire point of being long one leg and short the other. If either strategy's returns
correlate meaningfully with PAXG's own return, some of what looks like "arb P&L" is actually
disguised directional exposure, not a clean mispricing capture.


In [ ]:
paxg_ret = np.log(df["paxg_close"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_gold": [strategies["pure_mean_reversion"].corr(paxg_ret)],
    "trend_filtered_vs_gold": [strategies["trend_filtered_mean_reversion"].corr(paxg_ret)],
}, index=["correlation_with_paxg_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?

Section 6's returns are gross of fees. Each "trade" there is a full round-trip on *two* legs (PAXG
spot + the perp), entered and exited — four fills total. OKX's standard taker fee is roughly 0.05%
per fill on both spot and perpetual products; even at that (relatively low, taker-only) rate, this
adds up fast on a strategy whose entire gross edge is a few percent.


In [ ]:
TAKER_FEE_PER_FILL = 0.0005  # 0.05%, a reasonable OKX baseline; real cost is often higher (spread, slippage)
FILLS_PER_ROUND_TRIP = 4  # enter PAXG, enter perp, exit PAXG, exit perp

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways & Next Steps

This run's actual findings (re-run the notebook periodically — every number below will shift as more
OKX history accumulates for this still-young contract):

- **The basis did not test as stationary in this sample** (Section 2's ADF test) — the core
  statistical assumption a mean-reversion strategy depends on didn't hold up here. That alone is a
  reason to treat the backtest as illustrative rather than validated, independent of everything else.
- **The trend/basis correlation came back real but in the opposite direction from the opening
  hypothesis** (Section 4): momentum-chasing perpetual flow predicted a *positive* correlation
  (perp richens on uptrends); the data showed a *negative* one instead. That's still evidence the two
  ideas connect — just not through the mechanism assumed going in. Worth digging into why before
  trusting it (see Section 4's printed interpretation for the specific number from this run).
- **In this sample, the trend filter didn't help** — it reduced both total return and Sharpe versus
  pure mean-reversion (Section 6), despite the real correlation found in Section 4. A filter can be
  directionally justified and still not pay off if it also blocks profitable trades along with the
  bad ones; that trade-off is what actually determines whether "combining the two ideas" creates
  alpha, not just whether they're statistically related.
- **Check Section 8 before getting excited about Section 6's Sharpe**: gross returns this small (a
  few percent over ~10 months) are exactly the kind of edge that real trading costs can erase
  entirely — compare `est_net_total_return` to `gross_total_return` directly above.
- **The funding-rate finding is itself notable** (Section 1): if funding came back at or near zero
  for most of the sample, this contract's basis isn't being pulled back by the usual crypto-perp
  funding-payment mechanism — convergence, if any, is coming from something else (direct arbitrage
  flow, market-maker inventory management, shared price discovery). Worth understanding before
  sizing a real position around "funding will do the work."
- **PAXG redemption mechanics aren't modeled**: real-world arbitrage that keeps PAXG anchored to
  gold requires whole-ounce redemption lots and on-chain gas costs that a small backtest position
  wouldn't actually clear.
- **Extend the comparison**: run the same trend-filter idea against other gold-tracking pairs (e.g.
  PAXG vs. XAUT/Tether Gold, or a traditional COMEX gold futures feed if you have access) to see
  whether Section 4's trend-basis relationship is a property of *this specific* synthetic-perp
  mechanism or something more general about gold-tracking instruments.
- **Robustness**: the overlapping sample is short (~10 months) — there isn't enough history yet for
  a meaningful out-of-sample split. Re-run this notebook as more data accumulates before treating
  any of it as an actual edge.
